In [1]:
!pip install osmnx
import osmnx as ox
import geopandas as gpd
import pandas as pd

# Define countries
countries = ["Kenya", "Tanzania", "Uganda", "Ethiopia"]
all_rails = []

print("Starting data download from OSM... this may take a few minutes.")

for country in countries:
    try:
        # Fetch data
        gdf = ox.features_from_place(country, {"railway": "rail"})
        gdf['country'] = country

        # Project for KM calculation
        gdf = gdf.to_crs("ESRI:102022")
        gdf['length_km'] = gdf.geometry.length / 1000

        def categorize(row):
            gauge = str(row.get('gauge', 'unknown'))
            if '1435' in gauge: return 'Standard Gauge (SGR)'
            if '1000' in gauge: return 'Meter Gauge (MGR)'
            return 'Other/Legacy'

        gdf['rail_type'] = gdf.apply(categorize, axis=1)
        # Keep only necessary columns to save space
        all_rails.append(gdf[['country', 'rail_type', 'length_km', 'geometry']])
        print(f"Finished {country}")
    except Exception as e:
        print(f"Error in {country}: {e}")
        continue

# Combine and save
master_gdf = gpd.GeoDataFrame(pd.concat(all_rails, ignore_index=True)).to_crs("EPSG:4326")
master_gdf.to_parquet("railway_data.parquet")

# Download to your local machine
from google.colab import files
files.download("railway_data.parquet")

ModuleNotFoundError: No module named 'osmnx'